# Agent-Based Model: Structural Determinants of Infection
### Research Bazaar — JupyterLite Demo

This model simulates how **structural factors** — deprivation, housing crowding, and occupation type — drive differential infection risk across a population. It illustrates why standard epidemic models that assume homogeneous mixing underestimate risk for the most vulnerable groups.

**Run each cell in order using Shift + Enter.**

In [ ]:
# Cell 1 — Import libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
print('Libraries loaded successfully.')

In [ ]:
# Cell 2 — Create the population of agents
np.random.seed(42)
N = 500
GRID = 50

positions = np.random.randint(0, GRID, size=(N, 2)).astype(float)

deprivation = np.clip(np.random.beta(2, 3, N) * 10, 0, 10)

# Essential workers correlated with deprivation
occupation = np.array([
    np.random.binomial(1, 0.65) if deprivation[i] > 6
    else np.random.binomial(1, 0.20)
    for i in range(N)
])

household_size = np.random.choice([1,2,3,4,5,6,7,8], N,
                                   p=[0.1,0.2,0.25,0.2,0.1,0.07,0.05,0.03])

# Spatial clustering — high deprivation and essential workers cluster together
high_dep_idx = np.where(deprivation > 6)[0]
positions[high_dep_idx] = positions[high_dep_idx] * 0.4 + np.array([30, 30])
essential_idx = np.where(occupation == 1)[0]
positions[essential_idx] = positions[essential_idx] * 0.5 + np.array([25, 25])
positions = np.clip(positions, 0, GRID - 1)

state = np.zeros(N, dtype=int)
centre_agents = np.argsort(np.sum((positions - GRID/2)**2, axis=1))[:5]
state[centre_agents] = 1
infection_timer = np.zeros(N, dtype=int)
infection_timer[centre_agents] = 1

print(f'Population created: {N} agents')
print(f'Essential workers: {occupation.sum()} ({occupation.mean()*100:.0f}%)')
print(f'Mean deprivation score: {deprivation.mean():.2f}')
print(f'Initially infected: {(state==1).sum()}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sc = axes[0].scatter(positions[:,0], positions[:,1],
                     c=deprivation, cmap='RdYlGn_r', s=8, alpha=0.7)
plt.colorbar(sc, ax=axes[0], label='Deprivation score')
axes[0].set_title('Agent population coloured by deprivation score', fontweight='bold')
axes[0].set_xlabel('Grid x')
axes[0].set_ylabel('Grid y')

colors = ['steelblue' if o==0 else 'tomato' for o in occupation]
axes[1].scatter(positions[:,0], positions[:,1], c=colors, s=8, alpha=0.7)
axes[1].set_title('Agent population coloured by occupation type', fontweight='bold')
axes[1].set_xlabel('Grid x')
from matplotlib.patches import Patch
axes[1].legend(handles=[Patch(color='steelblue', label='Work from home'),
                         Patch(color='tomato', label='Essential worker')])

plt.tight_layout()
plt.show()
print('Population visualised.')

In [ ]:
# Cell 3 — Structural transmission parameters
base_beta = 0.15
crowding_multiplier = 2.5
worker_multiplier = 2.0
deprivation_isolation = 0.4
INFECTIOUS_DAYS = 7
CONTACT_RADIUS = 2

print('Transmission parameters set:')
print(f'  Base transmission probability per contact: {base_beta*100:.0f}%')
print(f'  Crowding multiplier (household 5+):        x{crowding_multiplier}')
print(f'  Essential worker contact multiplier:       x{worker_multiplier}')
print(f'  Isolation probability - high deprivation:  {deprivation_isolation*100:.0f}%')
print(f'  Isolation probability - low deprivation:   80%')
print(f'  Infectious period:                         {INFECTIOUS_DAYS} days')

In [ ]:
# Cell 4 — Run the simulation
def run_simulation(positions, deprivation, occupation, household_size,
                   base_beta, crowding_multiplier, worker_multiplier,
                   deprivation_isolation, INFECTIOUS_DAYS, CONTACT_RADIUS,
                   structured=True, steps=100):

    N = len(positions)
    state = np.zeros(N, dtype=int)
    centre = np.argsort(np.sum((positions - 25)**2, axis=1))[:5]
    state[centre] = 1
    timer = np.zeros(N, dtype=int)
    timer[centre] = 1

    history = {'S': [], 'I': [], 'R': []}
    infected_ever = np.zeros(N, dtype=bool)
    infected_ever[centre] = True

    for step in range(steps):
        new_state = state.copy()
        new_timer = timer.copy()

        infected_idx = np.where(state == 1)[0]

        for i in infected_idx:
            if structured:
                iso_prob = deprivation_isolation if deprivation[i] >= 8 else 0.8
            else:
                iso_prob = 0.6
            if np.random.random() < iso_prob:
                continue

            dists = np.sqrt(np.sum((positions - positions[i])**2, axis=1))
            if structured:
                radius = CONTACT_RADIUS * (worker_multiplier if occupation[i] == 1 else 1.0)
            else:
                radius = CONTACT_RADIUS
            neighbours = np.where((dists < radius) & (state == 0))[0]

            for j in neighbours:
                if structured:
                    beta = base_beta
                    if household_size[j] >= 5:
                        beta *= crowding_multiplier
                else:
                    beta = base_beta

                if np.random.random() < beta:
                    new_state[j] = 1
                    new_timer[j] = 1
                    infected_ever[j] = True

            if timer[i] >= INFECTIOUS_DAYS:
                new_state[i] = 2
            else:
                new_timer[i] = timer[i] + 1

        state = new_state
        timer = new_timer
        history['S'].append((state == 0).sum())
        history['I'].append((state == 1).sum())
        history['R'].append((state == 2).sum())

    return history, infected_ever

print('Running uniform population simulation...')
history_uniform, infected_uniform = run_simulation(
    positions, deprivation, occupation, household_size,
    base_beta, crowding_multiplier, worker_multiplier,
    deprivation_isolation, INFECTIOUS_DAYS, CONTACT_RADIUS,
    structured=False)

print('Running structured population simulation...')
history_structured, infected_structured = run_simulation(
    positions, deprivation, occupation, household_size,
    base_beta, crowding_multiplier, worker_multiplier,
    deprivation_isolation, INFECTIOUS_DAYS, CONTACT_RADIUS,
    structured=True)

print('Both simulations complete.')

In [ ]:
# Cell 5 — Plot the epidemic curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

steps = range(100)

axes[0].plot(steps, history_uniform['S'], color='steelblue', linewidth=2, label='Susceptible')
axes[0].plot(steps, history_uniform['I'], color='tomato', linewidth=2.5, label='Infected')
axes[0].plot(steps, history_uniform['R'], color='seagreen', linewidth=2, label='Recovered')
axes[0].set_title('Uniform population\n(standard SIR assumption — everyone equal)', fontweight='bold')
axes[0].set_xlabel('Time step (days)')
axes[0].set_ylabel('Number of agents')
axes[0].legend()
axes[0].set_ylim(0, N)

axes[1].plot(steps, history_structured['S'], color='steelblue', linewidth=2, label='Susceptible')
axes[1].plot(steps, history_structured['I'], color='tomato', linewidth=2.5, label='Infected')
axes[1].plot(steps, history_structured['R'], color='seagreen', linewidth=2, label='Recovered')
axes[1].set_title('Structured population\n(deprivation + crowding + occupation modelled)', fontweight='bold')
axes[1].set_xlabel('Time step (days)')
axes[1].set_ylabel('Number of agents')
axes[1].legend()
axes[1].set_ylim(0, N)

plt.suptitle('Epidemic curves: Uniform vs Structurally Determined Population',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

peak_u = max(history_uniform['I'])
peak_s = max(history_structured['I'])
print(f'Peak infected — Uniform population:    {peak_u} agents ({peak_u/N*100:.1f}%)')
print(f'Peak infected — Structured population: {peak_s} agents ({peak_s/N*100:.1f}%)')

In [ ]:
# Cell 6 — Attack rate by deprivation quintile
df = pd.DataFrame({
    'deprivation': deprivation,
    'infected_uniform': infected_uniform,
    'infected_structured': infected_structured,
    'occupation': occupation,
    'household_size': household_size
})

df['deprivation_decile'] = pd.cut(df['deprivation'], bins=5,
                                   labels=['1 (Least\ndeprived)','2','3','4',
                                           '5 (Most\ndeprived)'])

attack_rates = df.groupby('deprivation_decile', observed=True).agg(
    attack_uniform=('infected_uniform', 'mean'),
    attack_structured=('infected_structured', 'mean'),
    n=('deprivation', 'count')
).reset_index()

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(attack_rates))
width = 0.35

bars1 = ax.bar(x - width/2, attack_rates['attack_uniform'] * 100,
               width, label='Uniform population', color='steelblue', alpha=0.8)
bars2 = ax.bar(x + width/2, attack_rates['attack_structured'] * 100,
               width, label='Structured population', color='tomato', alpha=0.8)

ax.set_xlabel('Deprivation quintile', fontsize=12)
ax.set_ylabel('Attack rate (%)', fontsize=12)
ax.set_title('Attack rate by deprivation quintile\nUniform vs Structured population',
             fontweight='bold', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(attack_rates['deprivation_decile'])
ax.legend(fontsize=11)
ax.set_ylim(0, 100)
ax.axhline(y=df['infected_structured'].mean()*100,
           color='tomato', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print('Attack rates by deprivation quintile (structured population):')
for _, row in attack_rates.iterrows():
    bar = '█' * int(row['attack_structured']*20)
    print(f"  {str(row['deprivation_decile']).replace(chr(10),' '):20s}: {row['attack_structured']*100:5.1f}%  {bar}")

print(f"\nOverall attack rate - Uniform:    {df['infected_uniform'].mean()*100:.1f}%")
print(f"Overall attack rate - Structured: {df['infected_structured'].mean()*100:.1f}%")
print('\nThe deprivation gradient above is the structural epi signature.')
print('Higher deprivation = higher attack rate, driven by crowding, occupation, and isolation barriers.')